# Improve DeepCAD

In this notebook the idea to improve DeepCAD by splitting up the CAD-sequences in smaller units will be explored.

__IMPORTANT__

- For this I disabled the sampling of point clouds to 2048 points, here I use the whole 8096 points

In [1]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply

In [2]:
def get_data(dataset, index):
    data = dataset[index]
    point_cloud = data['pc']
    sequence = np.asarray(data['tgt_vec'])
    return point_cloud, sequence

In [3]:
def copy_to_temp(dataset, idx):
    cad_seq_path = dataset.get_cad_seq_path(idx)
    pc_path = dataset.get_pc_path(idx)
    print(idx)
    print(pc_path)
    json_path = pc_path.replace("pc_cad", "cad_json")
    json_path = json_path.replace("ply", "json")
    print(pc_path, json_path)
    
    temp_dir = "../data/temporary"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    dest_path_cad_seq = os.path.join(temp_dir, os.path.basename(cad_seq_path))
    dest_path_pc = os.path.join(temp_dir, os.path.basename(pc_path))
    dest_path_json = os.path.join(temp_dir, os.path.basename(json_path))

    shutil.copy(cad_seq_path, dest_path_cad_seq)
    shutil.copy(pc_path, dest_path_pc)
    shutil.copy(json_path, dest_path_json)
    
    print(f"Copied {cad_seq_path} to {dest_path_cad_seq}")
    print(f"Copied {pc_path} to {dest_path_pc}")
    print(f"Copied {json_path} to {dest_path_json}")
    return dest_path_json

def change_keys(h5_file):
    """Changes the keys from 'vec' to 'out_vec' in order to be able to show the sample using show.py"""
    with h5py.File(h5_file, 'r+') as hf:

        if 'vec' in hf:
            data = hf['vec'][:]
            hf.create_dataset('out_vec', data=data)
            del hf['vec']
            print(f"Changed keys from 'vec' to 'out_vec' in {h5_file}")

def export2step(json_path):
    filter = True
    save_path = os.path.join(*json_path.split("/")[:-1], os.path.splitext(os.path.basename(json_path))[0] + '.step')

    with open(json_path, "r") as fp:
        data = json.load(fp)

    cad_seq = CADSequence.from_dict(data)
    cad_seq.normalize()
    shape = create_CAD(cad_seq)

    write_step_file(shape, save_path)
    return save_path

def step2stl(step_path):

    save_path = os.path.join(*step_path.split("/")[:-1], os.path.splitext(os.path.basename(step_path))[0] + '.stl')
    step_reader = STEPControl_Reader()
    step_reader.ReadFile(step_path)
    step_reader.TransferRoots()
    shape = step_reader.OneShape()

    BRepMesh_IncrementalMesh(shape, 0.1)

    stl_writer = StlAPI_Writer()
    stl_writer.Write(shape, save_path)
    print(f"Wrote stl file to {save_path}")

def visualize_gt(dataset, idx):
    dest_path_h5 = copy_to_temp(dataset, idx)
   # change_keys(dest_path_h5)
    step_path = export2step(dest_path_h5)
    step2stl(step_path)

In [4]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [5]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0],
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0]
]
eos_row = [3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]

In [6]:
seq_np = np.array(seq, dtype=np.float32)
num_pad_rows = 60 - seq_np.shape[0]
pad_array = np.tile(eos_row, (num_pad_rows, 1))
seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
seq_len = seq_np_pad[:,0].tolist().index(3)
cad_seq = CADSequence.from_vector(seq_np_pad, is_numerical=True)
custom_shape = create_CAD(cad_seq)
write_step_file(custom_shape, "a.step")
step2stl("a.step")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : a.step(655 ents)  Write  Done
Wrote stl file to a.stl


In [7]:
out_pc = CADsolid2pc(custom_shape, 8096, "aha")
write_ply(out_pc, "aha.ply")

In [8]:
seq_np_pad.shape

(60, 17)

In [9]:
visualize_gt(dataset, i)

NameError: name 'i' is not defined

## START

We will use the below sequence to create a minimum working example.

In [10]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

#### 1. Split CAD-sequence into extrusions

In [11]:
def seq2shape(seq):
    cad_seq = CADSequence.from_vector(seq, is_numerical=True)
    shape = create_CAD(cad_seq)
    return shape

In [12]:
def shape2cad(shape, name=None):
    if name is None:
        name = "test"
    write_step_file(shape, os.path.join("examples", name + ".step"))
    step2stl(os.path.join("examples",name + ".step"))

In [13]:
def seq2CAD(seq, name=None):
    """Takes (60,17) sequence and turns it to stl file."""
    shape = seq2shape(seq)
    shape2cad(shape, name=name)

In [14]:
def pad_seq(seq):
    """Takes custom sequence (N,17) and pads it to (60,17)"""
    eos_row = [3] + 16 * [-1]
    seq_np = np.array(seq, dtype=np.float32)
    num_pad_rows = 60 - seq_np.shape[0]
    pad_array = np.tile(eos_row, (num_pad_rows, 1))
    seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
    return seq_np_pad
    

In [15]:
def split_and_pad_sequence_by_extrusion(matrix, delimiter=5):
    """Takes (60,17) sequence and splits it by the extrusions and pads it and returns a (60,17) for each extrusion""" 
    matrix = np.array(matrix)
    assert matrix.shape == (60, 17), "Input must be (60, 17)"

    commands = matrix[:,0]
    split_indices = []
    start_idx = 0

    # Find split points
    for idx, val in enumerate(commands):
        if val == delimiter:
            split_indices.append((start_idx, idx))
            start_idx = idx + 1

    # Split and pad
    output = []
    for start, end in split_indices:
        length = 59 - (end - start)
        pad_row = [[3] + 16 * [-1]] * length
        new_matrix = matrix[start:end+1]
       
        pad_matrix = np.vstack([new_matrix, pad_row])
        output.append(pad_matrix)
    return output

In [113]:
def seq2pc(seq, nr_points=8096, name=None):
    shape = seq2shape(seq)
    if name is None:
        name = "test"
    out_pc = CADsolid2pc(shape, nr_points, name)
    write_ply(out_pc, os.path.join("examples",name + ".ply"))
    return out_pc

In [16]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 64, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [17]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 64, 64, 192, 128, 3, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [18]:
sequence = pad_seq(seq)

In [19]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)


In [20]:
index = 0
data = dataset[index]
sequence = data["tgt_vec"].numpy()
print(sequence[:,0])

[4 0 0 0 4 2 4 2 4 2 4 2 4 2 5 4 2 5 4 2 5 4 2 5 4 2 5 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]


In [21]:
shape = seq2CAD(sequence, "round")
seq2pc(sequence, name="round")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
Wrote stl file to examples/round.stl
** WorkSession : Sending all data
 Step File Name : examples/round.step(1082 ents)  Write  Done


In [22]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)

In [23]:
for i, ext_seq in enumerate(extrusion_splits_seq):
    seq2CAD(ext_seq, name=str(i) + "_test")
    seq2pc(ext_seq, name=str(i) + "_test")
    


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/0_test.step(722 ents)  Write  Done
Wrote stl file to examples/0_test.stl
Wrote stl file to examples/1_test.stl

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/1_test.step(148 ents)  Write  Done

***************************************************

## Boolean

We can segment point clouds by their extrusions by iteratively building the model extrusion per extrusion and saving the labels. However there are problems with cut extrusions and interior walls in the final point cloud.

In [17]:
def combine_extrusions(base, extrusion):
    """Appends extrusion to base. Both are (60,17) from split_and_pad_sequence_by_extrusion()."""
    start = np.where(base == 3)[0][0]
    end = np.where(extrusion == 3)[0][0]
    combined = base
    combined[start:start+end, :] = extrusion[0:end, :]
    return combined

In [18]:
def extrusion_up_to(sequence, end_ext_idx):
    """Returns sequence up to the specified extrusion (inclusive). Extrusions are zero indexed."""
    extrusion = split_and_pad_sequence_by_extrusion(sequence)
    base = extrusion[0]
    for i in range(end_ext_idx):
        combined = combine_extrusions(base, extrusion[i+1])
        base = combined

    return base

In [19]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 0, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [92]:
# Cube with cylinder on top 
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 64, 128, 2, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [21]:
import mesh_to_sdf
import trimesh

def remove_inner_points_with_sdf(points, mesh_path, threshold=0.0):
    """
    Removes all points that are inside the mesh by checking signed distance.
    :param points: Nx3 numpy array
    :param mesh_path: path to final STL mesh
    :param threshold: values < threshold are considered 'inside'
    :return: filtered_points, indices
    """
    mesh = trimesh.load(mesh_path)

    # Ensure the mesh is watertight
    if not mesh.is_watertight:
        print("Warning: Mesh is not watertight. SDF results may be unreliable.")

    sdf_values = mesh_to_sdf.mesh_to_sdf(mesh, points, surface_point_method='sample', sign_method='normal')

    keep_mask = sdf_values >= threshold  # only keep points outside or on surface
    return points[keep_mask], np.where(keep_mask)[0]


In [94]:
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Cut
def segment_pc_into_extrusion(seq, nr_point=8096):
    extrusions = split_and_pad_sequence_by_extrusion(seq)
    all_points = []
    all_labels = []

    for i in range(len(extrusions)):

        if i > 0:
            print("\n prev")
            for a in extrusion_up_to(seq, i-1):
                print(a[0], end='')
            shape_prev = seq2shape(extrusion_up_to(seq, i-1))
        else:
            shape_prev = None

        print("\n curr")
    #    for a in extrusion_up_to(seq, i):
     #       print(a[0], end='')

        shape_curr = seq2shape(extrusion_up_to(seq, i))
        
    
        if shape_prev is not None:
            shape_diff = BRepAlgoAPI_Cut(shape_curr, shape_prev).Shape()
        else:
            shape_diff = shape_curr
       # if shape_prev is not None:
        #    shape2cad(shape_prev, f"shape_prev{i}")
   #     shape2cad(shape_curr, f"shape_curr{i}")
    #    shape2cad(shape_diff, f"shape_diff{i}")
        pc = CADsolid2pc(shape_diff, nr_point)
        labels = np.full((pc.shape[0],), i, dtype=int)
    
        all_points.append(pc)
        all_labels.append(labels)
        
    all_points = np.concatenate(all_points, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    return all_points, all_labels

In [84]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [85]:
Not correct: 0, 3
Failed: 1, 2, 4

SyntaxError: invalid syntax (484727421.py, line 1)

In [89]:
data = get_data(dataset, 10)
for a in data[1]:
    print(a[0], end='   ')
print("")
for i, a in enumerate(data[1]):
    print(a[15], end=' ')
    if a[15] == 2:
        data[1][i][15] = 0
seq2CAD(data[1], "dataset")

4   0   0   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   
-1 -1 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 
*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/dataset.step(960 ents)  Write  Done
Wrote stl file to examples/dataset.stl


In [87]:
for a in data[1]:
    print(a[15], end=' ')

-1 -1 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 

In [96]:
#all_points, all_labels = segment_pc_into_extrusion(pad_seq(seq), 8096)
final_shape = seq2shape(pad_seq(seq))
shape2cad(final_shape, "final")
print(type(all_points), all_points.shape)
surface_points, surface_idx = remove_inner_points_with_sdf(all_points, "examples/final.stl")
surface_labels = all_labels[surface_idx]


*******************************************************************
******        Statistics on Transfer (Write)                 ******
Wrote stl file to examples/final.stl
<class 'numpy.ndarray'> (8096, 3)

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/final.step(814 ents)  Write  Done


In [33]:
import open3d as o3d

def visualize_labeled_pc(points, labels):
    max_label = labels.max() + 1
    colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])


In [34]:
visualize_labeled_pc(surface_points, surface_labels)

/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_7608/263153949.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [51]:
surface_points.shape

(18531, 3)

In [52]:
surface_labels.shape

(18531,)

## New idea

In [141]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 0, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 64, 128, 2, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]
sequence = pad_seq(seq)

In [142]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)

In [143]:
complete_pc = seq2pc(sequence, name=str(i) + "complete")
seq2CAD(sequence, "full")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
Wrote stl file to examples/full.stl
** WorkSession : Sending all data
 Step File Name : examples/full.step(977 ents)  Write  Done


In [134]:
point_clouds = []
for i, ext_seq in enumerate(extrusion_splits_seq):
    pc = seq2pc(ext_seq, name=str(i) + "_test")
    point_clouds.append(pc)
merged_pc = np.concatenate(point_clouds, axis=0)
    
labels = []
for i, pc in enumerate(point_clouds):
    labels.append(np.full((pc.shape[0],), i, dtype=int))
merged_labels = np.concatenate(labels, axis=0)
  

In [135]:
from scipy.spatial import cKDTree

def filter_by_nearest_neighbor(pc_A, pc_B, epsilon=0.5):
    """
    Filters point cloud B using nearest neighbors from point cloud A.
    Keeps only points in B that are within `epsilon` distance to any point in A.
    
    :param pc_A: Nx3 point cloud (final model, no labels)
    :param pc_B: Mx3 point cloud (labeled extrusions)
    :return: filtered_pc_B, mask (M,), where mask[i] = True if point i is kept
    """
    tree = cKDTree(pc_A)
    distances, _ = tree.query(pc_B, k=1)

    mask = distances < epsilon
    return pc_B[mask], mask


In [136]:
# Assuming:
# pc_A: final model point cloud (Nx3)
# pc_B: merged extrusion point cloud (Mx3)
# labels_B: labels for each point in pc_B (Mx1)

filtered_pc, mask = filter_by_nearest_neighbor(complete_pc, merged_pc, epsilon=0.01)
filtered_labels = merged_labels[mask]

print("Remaining points:", filtered_pc.shape[0])


Remaining points: 3926


In [137]:
visualize_labeled_pc(filtered_pc, filtered_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [123]:
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = plt.cm.tab10(labels / labels.max())[:, :3]  # normalize
    pcd.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd])

visualize_labeled_pc(merged_pc, merged_labels)


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [160]:
def print_sequence(sequence):
    """Input: (NxM) sequence matrix"""
    for a in sequence:
        print(a[0], end='')
        if a[0] == 3:
            break
        if a[0] == 5:
            print(" ", end="")
    print("")
    

In [165]:
def get_labled_pc_per_ext(sequence):
    """Input: sequence (60x17), Output: As many point clouds as there are extrusions, merged, each with a label"""
    extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)
    point_clouds = []
    for i, ext_seq in enumerate(extrusion_splits_seq):
        pc = seq2pc(ext_seq, name=str(i) + "_test")
        point_clouds.append(pc)
    merged_pc = np.concatenate(point_clouds, axis=0)
        
    labels = []
    for i, pc in enumerate(point_clouds):
        labels.append(np.full((pc.shape[0],), i, dtype=int))
    merged_labels = np.concatenate(labels, axis=0)
    return merged_pc, merged_labels

In [185]:
def save_pc(pc, path):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [261]:
from plyfile import PlyData, PlyElement
import numpy as np

def export_xyz_label_ply(points, labels, fname="examples/labeled_points.ply"):
    """
    Write a PLY with an extra uchar 'label' property.
    """
    points = np.asarray(points, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.uint32)          # uint8/16/32 all fine

    ply_vertices = np.empty(len(points), 
                            dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"), 
                                   ("label", "u4")])
    ply_vertices["x"] = points[:, 0]
    ply_vertices["y"] = points[:, 1]
    ply_vertices["z"] = points[:, 2]
    ply_vertices["label"] = labels

    el = PlyElement.describe(ply_vertices, "vertex")
    PlyData([el], text=True).write(fname)
    print(f"Saved {fname}")


### START

In [252]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [267]:
data = get_data(dataset, 9)
print_sequence(data[1])

425 425 3


In [268]:
seq2CAD(data[1], "gt_cad")


*******************************************************************
******        Statistics on Transfer (Write)                 ******
Wrote stl file to examples/gt_cad.stl

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/gt_cad.step(238 ents)  Write  Done


In [269]:
complete_pc = seq2pc(data[1], name="gt_pc")
merged_pc, merged_labels = get_labled_pc_per_ext(data[1])
print(merged_pc.shape)

(16192, 3)


In [270]:
visualize_labeled_pc(merged_pc, merged_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [271]:
filtered_pc, mask = filter_by_nearest_neighbor(complete_pc, merged_pc, epsilon=0.005)
filtered_labels = merged_labels[mask]

In [272]:
save_pc(filtered_pc, "examples/filtered_pc.ply")

In [273]:
export_xyz_label_ply(filtered_pc, filtered_labels)

Saved examples/labeled_points.ply


In [ ]:
visualize_labeled_pc(filtered_pc, filtered_labels)